# 第 3 周练习：生成合成数据（Synthetic Data）

本练习对应课程 **Week 3**：用 Hugging Face 开源因果语言模型（Causal LM）在本地/Colab 生成结构化合成数据，并用 **Gradio** 做交互界面。

## 练习目标

1. 构建能生成数据集的模型流水线（输出 JSON、CSV 或纯文本）。
2. 切换多种 Hugging Face 指令微调（Instruct）模型与提示词，观察输出差异。
3. 用 Gradio UI 让用户选择模型、风格、条数，并一键生成；进阶版还支持下载文件。

## 怎么跑

1. **请在 Google Colab 运行本 notebook**（依赖 `google.colab.userdata` 与 GPU 量化）。
2. 在 Colab Secrets 中添加 Hugging Face token：键名必须是 `HF_TOKEN`。
3. 从上到下依次运行：安装依赖 → 登录 HF → 定义模型/规则 → 加载与生成函数 → 启动 Gradio。
4. 若显存紧张，优先选较小的模型（如 SmolLM2）；量化（4-bit）可显著省显存。


In [ ]:
# ========== 安装依赖：Colab 上第一次跑需要 ==========
# %pip：在 notebook 里安装包（逻辑等价于 shell 的 pip install）
# bitsandbytes：4-bit 量化；accelerate：设备映射；transformers 固定版本以免 API 漂移
# sentencepiece：部分分词器需要；gradio：Web UI；torch：张量与推理
%pip install -q --upgrade bitsandbytes accelerate "transformers==4.57.6" sentencepiece gradio torch


In [ ]:
# ========== 导入：后面生成合成数据要用的工具箱 ==========

# 标准库 os：文件路径、关闭文件描述符等
import os
# tempfile：生成临时文件，供 Gradio 下载
import tempfile
# lru_cache：缓存已加载的 tokenizer/model，避免重复下载占显存
from functools import lru_cache

# Colab 专用：从 Secrets 读取 HF_TOKEN，避免把密钥写进代码
from google.colab import userdata
# Hugging Face Hub 登录：用 token 拉取 gated 模型
from huggingface_hub import login
# AutoTokenizer / AutoModelForCausalLM：按 model_id 自动加载分词器与因果语言模型
# BitsAndBytesConfig：配置 4-bit 量化参数
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
# torch：设备、dtype、推理模式
import torch
# Gradio：快速搭交互界面
import gradio as gr


In [ ]:
# ========== 登录 Hugging Face：才能下载部分受控模型 ==========

# 从 Colab Secrets 取出名为 HF_TOKEN 的密钥（键名不要改）
hf_token = userdata.get('HF_TOKEN')
# login：把 token 交给 huggingface_hub；add_to_git_credential=True 方便后续 git 操作
login(hf_token, add_to_git_credential=True)


In [ ]:
# ========== 可选模型与输出格式规则（发给模型的英文指令保持原样） ==========

# MODELS：界面显示名 → Hugging Face model_id（字符串是可运行标识，勿改）
MODELS = {
    "Llama-3.2-3B-Instruct": "meta-llama/Llama-3.2-3B-Instruct",
    "SmolLM2-1.7B-Instruct": "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "Qwen2-1.5B-Instruct": "Qwen/Qwen2-1.5B-Instruct",
}

# FORMAT_RULES：按输出格式附加到 system prompt 的硬性约束（prompt 文本不翻译）
FORMAT_RULES = {
    "JSON": "Return a JSON array containing exactly the requested number of objects with consistent fields tailored to the context. No explanations.",
    "CSV": "Return a CSV document with a header row and the requested number of data rows aligned to the context. No explanations.",
    "Raw Text": "Return the requested number of short prose entries separated by blank lines that reflect the context. No explanations.",
}


In [ ]:
# ========== 量化配置：用 4-bit 把大模型塞进有限显存 ==========

# BitsAndBytesConfig：bitsandbytes 的量化配置对象（Quantization）
def get_quantization_config():
    # 返回可传给 from_pretrained(..., quantization_config=...) 的配置
    return BitsAndBytesConfig(
        # load_in_4bit：权重量化到 4 bit，显著降显存
        load_in_4bit=True,
        # 双重量化：再压缩量化常数，进一步省内存
        bnb_4bit_use_double_quant=True,
        # 计算时用 bfloat16：速度与数值稳定性的折中
        bnb_4bit_compute_dtype=torch.bfloat16,
        # nf4：常见 4-bit 量化类型（NormalFloat4）
        bnb_4bit_quant_type="nf4",
    )


In [ ]:
# ========== 加载分词器 + 模型：带 LRU 缓存，同 model_id 只加载一次 ==========

# lru_cache：相同 (model_id, use_quant) 直接复用，避免反复占显存
@lru_cache(maxsize=6)
def load_text_components(model_id: str, use_quant: bool):
    # 从 Hub 拉取分词器；trust_remote_code=True 允许模型自定义代码
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    # 有的模型没有 pad_token：用 eos_token 顶上，避免 generate 时报错
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 有 GPU 且用户勾选量化：走 4-bit 路径
    if use_quant and torch.cuda.is_available():
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            # device_map="auto"：accelerate 自动把层分到可用设备
            device_map="auto",
            quantization_config=get_quantization_config(),
            trust_remote_code=True,
        )
    else:
        # 无量化：有 GPU 用 float16，否则 CPU float32
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto" if torch.cuda.is_available() else None,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            trust_remote_code=True,
        )
    # eval()：关闭 dropout 等训练行为，进入推理模式
    model.eval()
    return tokenizer, model


In [ ]:
# ========== 风格模板：把 UI 上的风格选项映射成英文写作指令 ==========

# STYLE_TEMPLATES：键是 UI 选项；值会拼进 user prompt（影响生成风格，勿改字符串）
STYLE_TEMPLATES = {
    "Concise": "Keep each record brief and to the point.",
    "Detailed": "Include rich, realistic detail in each record.",
    "Diverse": "Maximize variety across records (names, values, categories).",
    "Technical": "Use precise, technical language where appropriate.",
    "Balanced": "Mix clarity and variety without being verbose.",
}


In [ ]:
# ========== 组装 chat messages：system 定规则，user 带上下文与条数 ==========

def build_text_messages(
    style: str,
    context: str,
    return_format: str,
    record_count: int,
) -> list:
    # 空上下文时给一个通用场景，避免 prompt 缺字段
    context_value = (context or "general purpose scenario").strip()
    # 风格缺省为 Balanced
    style_value = (style or "Balanced").strip()
    # 查风格说明；未知风格回退 Balanced
    style_instruction = STYLE_TEMPLATES.get(style_value, STYLE_TEMPLATES["Balanced"])
    # 按输出格式取出 FORMAT_RULES 中的硬性指令
    directive = FORMAT_RULES.get(return_format, FORMAT_RULES["JSON"])

    # system_prompt：角色 + 格式约束（英文 prompt 保持原样，改译会改变模型行为）
    system_prompt = (
        "You generate synthetic datasets that are high quality, diverse, and free of personally identifiable information. "
        + directive
        + " Ensure outputs are consistent in structure and avoid any explanation or commentary."
    )
    # user_prompt：具体任务——上下文、风格、条数、格式
    user_prompt = (
        f"Context: {context_value}\n"
        f"Style: {style_instruction}\n"
        f"Generate exactly {record_count} records. Output format: {return_format}."
    )
    # 返回 OpenAI/HF chat 模板常见的 messages 列表
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]


In [ ]:
# ========== 核心生成：加载模型 → 编码 messages → generate → 只解码新 token ==========

def generate_text_data(
    model_choice: str,
    style: str,
    context: str,
    return_format: str,
    quantize: bool,
    record_count: int,
) -> str:
    # 把 UI 显示名映射成 Hugging Face model_id
    model_id = MODELS.get(model_choice)
    # 未知选项时返回错误文案（这是影响 UI 判断的字符串，保持英文原样）
    if not model_id:
        return "Error: Unknown model selected."

    # 加载（或从缓存取）分词器与模型
    tokenizer, model = load_text_components(model_id, bool(quantize))
    # 拼出 system/user messages
    messages = build_text_messages(style, context, return_format, int(record_count))

    # 优先用 chat template：把 messages 编成模型期望的对话格式张量
    if hasattr(tokenizer, "apply_chat_template"):
        inputs = tokenizer.apply_chat_template(
            messages,
            return_tensors="pt",
            # add_generation_prompt：在末尾加上「助手开始回答」标记
            add_generation_prompt=True,
        )
    else:
        # 没有 chat template 时退化为只编码最后一条 user 文本
        prompt = messages[-1]["content"]
        inputs = tokenizer(prompt, return_tensors="pt")

    # 模型参数所在设备（GPU/CPU），输入必须搬到同一设备
    device = next(model.parameters()).device
    if isinstance(inputs, dict):
        input_ids = inputs["input_ids"].to(device)
    else:
        # apply_chat_template 有时直接返回 Tensor
        input_ids = inputs.to(device) if hasattr(inputs, "to") else torch.tensor([inputs], device=device)
    # 保证 batch 维存在：形状 [batch, seq]
    if input_ids.dim() == 1:
        input_ids = input_ids.unsqueeze(0)
    # attention_mask：全 1 表示这些位置都参与注意力
    attention_mask = torch.ones_like(input_ids, device=device)

    # inference_mode：关闭梯度，推理更快、更省内存
    with torch.inference_mode():
        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            # 最多新生成 1024 个 token
            max_new_tokens=1024,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.05,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    # 只取「新生成」的那段：去掉 prompt 对应的前缀 token
    new_tokens = generated[:, input_ids.shape[-1] :]
    # batch_decode 后取第一条样本
    text = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0]
    return text.strip()


In [ ]:
# ========== Gradio UI（基础版）：选模型/格式/风格 → 生成文本预览 ==========

# gr.Blocks：自定义布局的 Gradio 应用
with gr.Blocks(title="Generating Synthetic Data") as demo:
    # 标题与简短说明（界面文案保持英文原样）
    gr.Markdown("## Generating Synthetic Data")
    gr.Markdown("Choose a Hugging Face model and generate datasets in JSON, CSV, or Raw Text.")

    # 一行三列：模型、输出格式、风格
    with gr.Row():
        model_choice = gr.Dropdown(
            choices=list(MODELS.keys()),
            value="SmolLM2-1.7B-Instruct",
            label="Model",
        )
        return_format = gr.Dropdown(
            choices=["JSON", "CSV", "Raw Text"],
            value="JSON",
            label="Output format",
        )
        style = gr.Dropdown(
            choices=list(STYLE_TEMPLATES.keys()),
            value="Balanced",
            label="Style",
        )

    # 上下文：描述想要的数据集领域/字段
    context_input = gr.Textbox(
        label="Context",
        lines=4,
        placeholder="e.g. Product catalog for an online electronics store: name, category, price, sku, in_stock",
    )
    # 生成条数滑块：1–20
    record_count = gr.Slider(1, 20, value=5, step=1, label="Number of records")

    generate_btn = gr.Button("Generate")
    text_output = gr.Textbox(label="Generated data", lines=16)

    # 点击回调：量化固定为 True（本版 UI 未暴露开关）
    def run_generate(model_choice, style, context, return_format, record_count):
        return generate_text_data(model_choice, style, context, return_format, True, record_count)

    # 把按钮与输入输出绑在一起
    generate_btn.click(
        fn=run_generate,
        inputs=[model_choice, style, context_input, return_format, record_count],
        outputs=text_output,
    )

# share=True：生成公网临时链接；debug=True：出错时显示更多信息
demo.launch(share=True, debug=True)


In [ ]:
# ========== 把生成结果落到临时文件，供 Gradio File 组件下载 ==========

def save_generated_to_file(text: str, return_format: str):
    """Save generated text to a file and return its path for download. Returns None if error or empty."""
    # 空文本或错误前缀：不创建文件（Error: 前缀与 generate_text_data 约定一致）
    if not (text and text.strip()) or text.startswith("Error:"):
        return None
    # 按格式选后缀
    ext = {"JSON": ".json", "CSV": ".csv", "Raw Text": ".txt"}.get(return_format, ".txt")
    # mkstemp：创建安全的临时文件，返回 (fd, path)
    fd, path = tempfile.mkstemp(suffix=ext)
    try:
        # fdopen：用文件描述符写出 UTF-8 文本
        with os.fdopen(fd, "w", encoding="utf-8") as f:
            f.write(text)
        return path
    except Exception:
        # 写出失败时尽量关掉 fd，避免泄漏
        try:
            os.close(fd)
        except Exception:
            pass
        return None


In [ ]:
# ========== Gradio UI（进阶版）：预览 + 文件下载 ==========

with gr.Blocks(title="Generating Synthetic Data") as demo:
    gr.Markdown("## Generating Synthetic Data")
    gr.Markdown("Choose a Hugging Face model and generate datasets in JSON, CSV, or Raw Text.")

    with gr.Row():
        model_choice = gr.Dropdown(
            choices=list(MODELS.keys()),
            value="SmolLM2-1.7B-Instruct",
            label="Model",
        )
        return_format = gr.Dropdown(
            choices=["JSON", "CSV", "Raw Text"],
            value="JSON",
            label="Output format",
        )
        style = gr.Dropdown(
            choices=list(STYLE_TEMPLATES.keys()),
            value="Balanced",
            label="Style",
        )

    context_input = gr.Textbox(
        label="Context",
        lines=4,
        placeholder="e.g. Product catalog for an online electronics store: name, category, price, sku, in_stock",
    )
    record_count = gr.Slider(1, 20, value=5, step=1, label="Number of records")

    generate_btn = gr.Button("Generate")
    # 预览文本框
    text_output = gr.Textbox(label="Generated data (preview)", lines=16)
    # 文件下载组件
    file_output = gr.File(label="Download file")

    def run_generate(model_choice, style, context, return_format, record_count):
        # 先生成文本，再落盘；两个返回值分别喂给 Textbox 与 File
        text = generate_text_data(model_choice, style, context, return_format, True, record_count)
        file_path = save_generated_to_file(text, return_format)
        return text, file_path

    generate_btn.click(
        fn=run_generate,
        inputs=[model_choice, style, context_input, return_format, record_count],
        outputs=[text_output, file_output],
    )

# 启动应用；share=True 便于在 Colab 外访问
demo.launch(share=True)
